# External Framework Validation — SDMetrics (Bank Marketing)

Runs in the isolated `tese_sdmetrics` conda environment. Quality and Diagnostic
reports plus individual column metrics and DCR-based privacy. Mirrors the
Dataset 1 SDMetrics notebook, adapted to the Bank feature space.

## 0. Configuration and paths

In [1]:
from pathlib import Path
import warnings
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore")

RANDOM_STATE = 123

# Folder with the CSV files, relative to this notebook (adjust if needed)
DATA_DIR = Path("data")

assert DATA_DIR.exists(), f"Folder not found: {DATA_DIR}"
print("Data folder:", DATA_DIR)

import sdmetrics
print("sdmetrics version:", sdmetrics.__version__)

Data folder: data
sdmetrics version: 0.28.2


## 1. Canonical column definitions (Bank Marketing)

In [2]:
TARGET = "y"

FEATURES = ['age','job','marital','education','default','housing','loan','contact',
            'month','day_of_week','campaign','previous','poutcome','emp.var.rate',
            'cons.price.idx','cons.conf.idx','euribor3m','nr.employed','was_contacted']

# Categorical columns: the 11 nominal categoricals (incl. the binary flag).
# Bank Marketing has NO ordinal columns, so there is a single, unambiguous
# categorical treatment (no Variant A/B, unlike the PAY_X columns of Dataset 1).
CAT_COLS = ['job','marital','education','default','housing','loan','contact',
            'month','day_of_week','poutcome','was_contacted']
NUM_COLS = [c for c in FEATURES if c not in CAT_COLS]

assert len(FEATURES) == 19, f"FEATURES should have 19, got {len(FEATURES)}"
assert TARGET not in FEATURES, "y must not be in FEATURES"
assert len(CAT_COLS) == 11 and len(NUM_COLS) == 8
print("FEATURES:", len(FEATURES), "| CAT:", len(CAT_COLS), "| NUM:", len(NUM_COLS))

ALL_COLS = FEATURES + [TARGET]
CAT_WITH_TARGET = CAT_COLS + [TARGET]

FEATURES: 19 | CAT: 11 | NUM: 8


## 2. Build the SDMetrics metadata dictionary

In [3]:
metadata = {"columns": {}}
for col in ALL_COLS:
    if col in CAT_WITH_TARGET:
        metadata["columns"][col] = {"sdtype": "categorical"}
    else:
        metadata["columns"][col] = {"sdtype": "numerical"}

n_cat = sum(1 for c in metadata["columns"].values() if c["sdtype"] == "categorical")
n_num = sum(1 for c in metadata["columns"].values() if c["sdtype"] == "numerical")
print(f"metadata: {n_cat} categorical, {n_num} numerical, {len(metadata['columns'])} total")

metadata: 12 categorical, 8 numerical, 20 total


## 3. Helper: align a dataframe to the canonical schema
Bank categoricals are strings already; the target is cast to string for stable
SDMetrics typing.

In [4]:
def align(df, name):
    d = df.copy()
    if TARGET not in d.columns:
        d[TARGET] = 1   # minority reference: all y == 1
    d = d[ALL_COLS]
    d[NUM_COLS] = d[NUM_COLS].astype("float64")
    # categorical (incl. target) as string labels for stable SDMetrics typing
    for c in CAT_WITH_TARGET:
        d[c] = d[c].astype(str)
    assert list(d.columns) == ALL_COLS
    print(f"{name:<16} -> {d.shape}")
    return d

## 4. Load the real and synthetic datasets

In [5]:
real      = align(pd.read_csv(DATA_DIR / "bank_real_minority_train.csv"),  "real (min)")
syn_ctgan = align(pd.read_csv(DATA_DIR / "bank_synthetic_ctgan_final.csv"), "syn ctgan")
syn_ctab  = align(pd.read_csv(DATA_DIR / "bank_synthetic_ctab_final.csv"),  "syn ctabgan+")

for d in (syn_ctgan, syn_ctab):
    assert list(d.columns) == list(real.columns)
print("\nColumns aligned.")

real (min)       -> (3248, 20)
syn ctgan        -> (22335, 20)
syn ctabgan+     -> (22335, 20)

Columns aligned.


## 5. Reconstruct the canonical split → holdout

In [6]:
df_raw = pd.read_csv(DATA_DIR / "bank-additional-full.csv", sep=';')

df_c = df_raw.drop(columns=['duration'])
df_c['was_contacted'] = (df_c['pdays'] != 999).astype(int)
df_c = df_c.drop(columns=['pdays'])
df_c[TARGET] = (df_c['y'] == 'yes').astype(int)
df_c['default'] = df_c['default'].replace({'yes': 'unknown'})

X = df_c.drop(columns=TARGET)
y = df_c[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE)

# verify reproduction against the stored training partition
real_full = pd.read_csv(DATA_DIR / "bank_real_train_full.csv")
train_recon = X_train.copy()
train_recon[TARGET] = y_train
train_recon = train_recon[real_full.columns].reset_index(drop=True)
# Compare VALUES, not strict dtypes (int32 vs int64 store the same 0/1 data)
pd.testing.assert_frame_equal(
    train_recon, real_full.reset_index(drop=True),
    check_dtype=False,   # values are compared, not dtypes (int32 vs int64 store the same 0/1 data)
)
print("Split reproduced bit for bit (values). OK.")

holdout = X_test.copy()
holdout[TARGET] = y_test
holdout_min = align(holdout[holdout[TARGET].astype(int) == 1], "holdout (min)")
print("Minority holdout:", holdout_min.shape)

Split reproduced bit for bit (values). OK.
holdout (min)    -> (1392, 20)
Minority holdout: (1392, 20)


## 6. Quality Report

In [7]:
from sdmetrics.reports.single_table import QualityReport

def quality(syn, name):
    rep = QualityReport()
    rep.generate(real, syn, metadata)
    props = rep.get_properties()
    overall = rep.get_score()
    print(f"{name}: overall = {overall:.4f}")
    print(props.to_string(index=False))
    row = {"model": name, "overall_quality": overall}
    for _, r in props.iterrows():
        row[r["Property"]] = r["Score"]
    return row, rep

q_ctgan, rep_q_ctgan = quality(syn_ctgan, "CTGAN")
print()
q_ctab, rep_q_ctab = quality(syn_ctab, "CTAB-GAN+")
quality_rows = [q_ctgan, q_ctab]

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 20/20 [00:00<00:00, 331.54it/s]|
Column Shapes Score: 89.61%

(2/2) Evaluating Column Pair Trends: |██████████| 190/190 [00:00<00:00, 286.42it/s]|
Column Pair Trends Score: 74.17%

Overall Score (Average): 81.89%

CTGAN: overall = 0.8189
          Property    Score
     Column Shapes 0.896114
Column Pair Trends 0.741703

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 20/20 [00:00<00:00, 341.86it/s]|
Column Shapes Score: 86.15%

(2/2) Evaluating Column Pair Trends: |██████████| 190/190 [00:00<00:00, 292.03it/s]|
Column Pair Trends Score: 65.49%

Overall Score (Average): 75.82%

CTAB-GAN+: overall = 0.7582
          Property    Score
     Column Shapes 0.861543
Column Pair Trends 0.654864


## 7. Diagnostic Report

In [8]:
from sdmetrics.reports.single_table import DiagnosticReport

def diagnostic(syn, name):
    rep = DiagnosticReport()
    rep.generate(real, syn, metadata)
    props = rep.get_properties()
    overall = rep.get_score()
    print(f"{name}: overall diagnostic = {overall:.4f}")
    print(props.to_string(index=False))
    row = {"model": name, "overall_diagnostic": overall}
    for _, r in props.iterrows():
        row[r["Property"]] = r["Score"]
    return row

d_ctgan = diagnostic(syn_ctgan, "CTGAN")
print()
d_ctab = diagnostic(syn_ctab, "CTAB-GAN+")
diagnostic_rows = [d_ctgan, d_ctab]

Generating report ...

(1/2) Evaluating Data Validity: |██████████| 20/20 [00:00<00:00, 740.34it/s]|
Data Validity Score: 100.0%

(2/2) Evaluating Data Structure: |██████████| 1/1 [00:00<00:00, 333.52it/s]|
Data Structure Score: 100.0%

Overall Score (Average): 100.0%

CTGAN: overall diagnostic = 1.0000
      Property    Score
 Data Validity 0.999987
Data Structure 1.000000

Generating report ...

(1/2) Evaluating Data Validity: |██████████| 20/20 [00:00<00:00, 888.96it/s]|
Data Validity Score: 100.0%

(2/2) Evaluating Data Structure: |██████████| 1/1 [00:00<00:00, 500.39it/s]|
Data Structure Score: 100.0%

Overall Score (Average): 100.0%

CTAB-GAN+: overall diagnostic = 1.0000
      Property  Score
 Data Validity    1.0
Data Structure    1.0


## 8. Individual column-shape metrics (triangulation)

In [9]:
from sdmetrics.single_column import (
    KSComplement, TVComplement, BoundaryAdherence,
    CategoryAdherence, RangeCoverage, CategoryCoverage,
)

def col_shapes(syn, name):
    ks  = np.mean([KSComplement.compute(real[c], syn[c]) for c in NUM_COLS])
    tv  = np.mean([TVComplement.compute(real[c], syn[c]) for c in CAT_WITH_TARGET])
    ba  = np.mean([BoundaryAdherence.compute(real[c], syn[c]) for c in NUM_COLS])
    ca  = np.mean([CategoryAdherence.compute(real[c], syn[c]) for c in CAT_WITH_TARGET])
    rc  = np.mean([RangeCoverage.compute(real[c], syn[c]) for c in NUM_COLS])
    cc  = np.mean([CategoryCoverage.compute(real[c], syn[c]) for c in CAT_WITH_TARGET])
    row = {"model": name, "KSComplement_num": ks, "TVComplement_cat": tv,
           "BoundaryAdherence": ba, "CategoryAdherence": ca,
           "RangeCoverage": rc, "CategoryCoverage": cc}
    print(f"{name}: KS={ks:.4f} TV={tv:.4f} Boundary={ba:.4f} "
          f"CatAdh={ca:.4f} RangeCov={rc:.4f} CatCov={cc:.4f}")
    return row

shape_rows = [col_shapes(syn_ctgan, "CTGAN"), col_shapes(syn_ctab, "CTAB-GAN+")]

CTGAN: KS=0.8604 TV=0.9200 Boundary=1.0000 CatAdh=1.0000 RangeCov=1.0000 CatCov=1.0000
CTAB-GAN+: KS=0.7775 TV=0.9176 Boundary=1.0000 CatAdh=1.0000 RangeCov=0.8334 CatCov=1.0000


## 9. Novelty: NewRowSynthesis

In [10]:
from sdmetrics.single_table import NewRowSynthesis

def novelty(syn, name):
    score = NewRowSynthesis.compute(
        real_data=real, synthetic_data=syn, metadata=metadata,
        numerical_match_tolerance=0.01,
        synthetic_sample_size=min(5000, len(syn)),
    )
    print(f"{name}: NewRowSynthesis = {score:.4f}")
    return {"model": name, "NewRowSynthesis": score}

novelty_rows = [novelty(syn_ctgan, "CTGAN"), novelty(syn_ctab, "CTAB-GAN+")]

CTGAN: NewRowSynthesis = 1.0000
CTAB-GAN+: NewRowSynthesis = 1.0000


## 10. SDMetrics privacy metrics (DCR-based)

Note: the minority holdout (1,392 records) is smaller than half of the
training reference (3,248), below the size SDMetrics recommends for
`DCROverfittingProtection`. The library's warning is silenced by
`warnings.filterwarnings("ignore")`; the score is therefore less precise.

In [12]:
from sdmetrics.single_table import DCRBaselineProtection, DCROverfittingProtection

def dcr_privacy(syn, name):
    base = DCRBaselineProtection.compute(
        real_data=real, synthetic_data=syn, metadata=metadata)
    over = DCROverfittingProtection.compute(
        real_training_data=real, synthetic_data=syn,
        real_validation_data=holdout_min, metadata=metadata)
    base_s = base["score"] if isinstance(base, dict) else base
    over_s = over["score"] if isinstance(over, dict) else over
    print(f"{name}: DCRBaseline={base_s:.4f}  DCROverfitting={over_s:.4f}")
    return {"model": name, "DCRBaselineProtection": base_s,
            "DCROverfittingProtection": over_s}

privacy_rows = [dcr_privacy(syn_ctgan, "CTGAN"), dcr_privacy(syn_ctab, "CTAB-GAN+")]

CTGAN: DCRBaseline=0.4150  DCROverfitting=0.6010
CTAB-GAN+: DCRBaseline=0.4171  DCROverfitting=0.6187


## 11. Consolidate and save

In [13]:
from functools import reduce

frames = []
for rows in (quality_rows, diagnostic_rows, shape_rows, novelty_rows, privacy_rows):
    frames.append(pd.DataFrame(rows).set_index("model"))

sdmetrics_all = reduce(lambda a, b: a.join(b), frames).reset_index()
sdmetrics_all.to_csv(DATA_DIR / "bank_sdmetrics_all_results.csv", index=False)
print("Saved: bank_sdmetrics_all_results.csv\n")

with pd.option_context("display.max_columns", None, "display.width", 200):
    print(sdmetrics_all.round(4).to_string(index=False))

Saved: bank_sdmetrics_all_results.csv

    model  overall_quality  Column Shapes  Column Pair Trends  overall_diagnostic  Data Validity  Data Structure  KSComplement_num  TVComplement_cat  BoundaryAdherence  CategoryAdherence  RangeCoverage  CategoryCoverage  NewRowSynthesis  DCRBaselineProtection  DCROverfittingProtection
    CTGAN           0.8189         0.8961              0.7417                 1.0            1.0             1.0            0.8604            0.9200                1.0                1.0         1.0000               1.0              1.0                 0.4150                    0.6010
CTAB-GAN+           0.7582         0.8615              0.6549                 1.0            1.0             1.0            0.7775            0.9176                1.0                1.0         0.8334               1.0              1.0                 0.4171                    0.6187


---